# ReTrace — Resumable Qwen3-8B Hidden-State Extraction

**Rerun instructions:** after a runtime disconnect, open this notebook and choose **Runtime → Run all**; remount Drive when prompted, then wait for the model to load. Completed shards are read from Drive and skipped automatically, so extraction resumes rather than starting over.

The notebook extracts ProcessBench, PRM800K and HaluEval in resumable batches. It saves step-level representations from layers `-8`, `-4` and `-1`, using mean and last-token pooling, plus token IDs and step/token alignment metadata. It intentionally does not save every token's hidden vector because that would create much larger files.

**Keep** `MyDrive/ReTrace/retrace_colab_data.zip` and `MyDrive/ReTrace/hidden_states_full`. Do not delete the output folder between sessions. The full corpus is large; repeat **Run all** across sessions until the completion report says no samples remain.

## 1. Mount Google Drive

In [17]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/ReTrace/hidden_states_full"
)

test_file = folder / "_write_test.txt"

try:
    test_file.write_text("test", encoding="utf-8")
    print("Original folder is writable.")
    test_file.unlink()
except OSError as e:
    print("Original folder is still not writable:", e)

Original folder is writable.


In [ ]:
import os

print("MyDrive exists:", os.path.exists("/content/drive/MyDrive"))
print("Top-level contents:")

for item in os.listdir("/content/drive/MyDrive"):
    print(item)

In [10]:
import os

base = "/content/drive/MyDrive/ReTrace"

print("Folder exists:", os.path.exists(base))

if os.path.exists(base):
    for item in os.listdir(base):
        print(item)

Folder exists: True
sys arch.svg
retrace_colab_data.zip
hidden_states_pilot
hidden_states_full


## 2. Restore and validate the processed datasets

In [11]:
from pathlib import Path
import csv
import json
import zipfile

DRIVE_ROOT = Path("/content/drive/MyDrive/ReTrace")
ZIP_PATH = DRIVE_ROOT / "retrace_colab_data.zip"
DATA_DIR = Path("/content/retrace_data")
OUTPUT_ROOT = DRIVE_ROOT / "hidden_states_full"
MODEL_NAME = "Qwen/Qwen3-8B"
MAX_LENGTH = 4096
SHARD_SIZE = 100
MAX_NEW_SAMPLES_PER_RUN = 5000
LAYER_IDS = [-8, -4, -1]

assert ZIP_PATH.exists(), f"Dataset archive not found: {ZIP_PATH}"
DATA_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH) as archive:
    for member in archive.namelist():
        normalized = member.replace("\\", "/")
        target = (DATA_DIR / normalized).resolve()
        if DATA_DIR.resolve() not in target.parents and target != DATA_DIR.resolve():
            raise ValueError(f"Unsafe archive path: {member}")
        if member.endswith("/"):
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as src, open(target, "wb") as dst:
                dst.write(src.read())

processed_dir = DATA_DIR / "processed"
splits_dir = DATA_DIR / "splits"
sample_split_path = processed_dir / "sample_splits.csv"
manifest_path = splits_dir / "split_manifest.json"

required_files = [
    processed_dir / "processbench.jsonl",
    processed_dir / "prm800k.jsonl",
    processed_dir / "halueval.jsonl",
    sample_split_path,
    manifest_path,
]
missing = [str(p) for p in required_files if not p.exists()]
assert not missing, f"Missing dataset files: {missing}"

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
with open(sample_split_path, newline="", encoding="utf-8") as f:
    split_rows = list(csv.DictReader(f))
split_by_id = {r["sample_id"]: r["split"] for r in split_rows}

print("Dataset samples in split file:", len(split_by_id))
print("Split fingerprint:", manifest.get("samples_fingerprint_sha1"))
print("Expected fingerprint: b7c4c450ffe69b2bc14c5db61caeaa5ca30b4154")
assert manifest.get("samples_fingerprint_sha1") == "b7c4c450ffe69b2bc14c5db61caeaa5ca30b4154", "Unexpected split fingerprint"
assert len(split_by_id) == 139682, f"Expected 139682 samples, found {len(split_by_id)}"
print("Dataset and split validation passed.")

Dataset samples in split file: 139682
Split fingerprint: b7c4c450ffe69b2bc14c5db61caeaa5ca30b4154
Expected fingerprint: b7c4c450ffe69b2bc14c5db61caeaa5ca30b4154
Dataset and split validation passed.


## 3. Install dependencies and verify the GPU

In [12]:
%pip -q install -U transformers accelerate bitsandbytes
import torch

assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime → Change runtime type → T4 GPU."
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 112.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.8 MB/s eta 0:00:00
PyTorch: 2.11.0+cu130
GPU: Tesla T4
GPU VRAM (GB): 15.64


## 4. Load Qwen3-8B

In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
print("Model loaded:", MODEL_NAME)
print("Allocated GPU memory (GB):", round(torch.cuda.memory_allocated(0) / 1e9, 2))

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen3-8B
Allocated GPU memory (GB): 6.09


## 5. Define compact extraction and resumable shard helpers

In [36]:
import gc
import re
import time
import json
import torch

SOURCE_FILES = {
    "processbench": "processbench.jsonl",
    "prm800k": "prm800k.jsonl",
    "halueval": "halueval.jsonl",
}


def extract_trajectory(sample, split_name):
    question = sample["question"]
    steps = sample["steps"]

    if not isinstance(steps, list) or not steps:
        raise ValueError("trajectory has no reasoning/response steps")

    prefix = f"Question: {question}\nReasoning:\n"
    step_texts = [
        f"Step {i + 1}: {text}\n"
        for i, text in enumerate(steps)
    ]
    full_text = prefix + "".join(step_texts)
    prefix_end = len(prefix)

    encoded = tokenizer(
        full_text,
        return_tensors="pt",
        return_offsets_mapping=True,
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    offsets = encoded.pop("offset_mapping")[0].tolist()
    token_ids = encoded["input_ids"][0].cpu()
    inputs = {k: v.to(model.device) for k, v in encoded.items()}

    # Character boundaries for reasoning steps.
    char_boundaries = []
    cursor = prefix_end

    for text in step_texts:
        char_boundaries.append((cursor, cursor + len(text)))
        cursor += len(text)

    # Find the tokens belonging to the question/prompt prefix.
    prefix_token_indices = [
        i for i, (a, b) in enumerate(offsets)
        if b > a and a < prefix_end and b <= prefix_end
    ]

    if not prefix_token_indices:
        raise ValueError("No valid tokens found in the question prefix")

    # The final prefix token is the state immediately before Step 1.
    prefix_last_index = prefix_token_indices[-1]

    # Verify that tokenization did not create a token crossing the
    # prefix/first-step boundary.
    for i, (a, b) in enumerate(offsets):
        if a < prefix_end < b:
            raise ValueError(
                "A token crosses the question/first-step boundary; "
                "review tokenization before extracting this sample."
            )

    # Map each step's character span to token indices.
    step_token_indices = []

    for start, end in char_boundaries:
        indices = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < end and b > start
        ]

        if not indices:
            raise ValueError(
                f"Step has no tokens after truncation; "
                f"MAX_LENGTH={MAX_LENGTH}"
            )

        step_token_indices.append(indices)

    with torch.inference_mode():
        result = model(
            **inputs,
            output_hidden_states=True,
            use_cache=False,
        )

    step_states = {}
    question_states = {}

    for layer_id in LAYER_IDS:
        hidden = result.hidden_states[layer_id][0]

        # Existing per-step states: preserve the original format.
        mean_vectors = []
        last_vectors = []

        for indices in step_token_indices:
            index_tensor = torch.tensor(
                indices, device=hidden.device
            )
            selected = hidden.index_select(0, index_tensor)

            mean_vectors.append(
                selected.float().mean(dim=0).half().cpu()
            )
            last_vectors.append(
                selected[-1].detach().half().cpu()
            )

        step_states[layer_id] = {
            "mean": torch.stack(mean_vectors),
            "last_token": torch.stack(last_vectors),
        }

        # NEW: question/prompt representation immediately before Step 1.
        prefix_indices_tensor = torch.tensor(
            prefix_token_indices,
            device=hidden.device,
        )
        prefix_hidden = hidden.index_select(
            0, prefix_indices_tensor
        )

        question_states[layer_id] = {
            # State at the final question/prompt token.
            "last_token": (
                hidden[prefix_last_index].detach().half().cpu()
            ),

            # Optional pooled representation of the entire prefix.
            "mean": prefix_hidden.float().mean(dim=0).half().cpu(),
        }

    artifact = {
        "sample_id": sample["sample_id"],
        "source": sample["source"],
        "problem_id": sample["problem_id"],
        "split": split_name,
        "question": question,
        "steps": steps,
        "step_labels": sample.get("step_labels"),
        "first_error": sample.get("first_error"),
        "answer": sample.get("answer"),
        "answer_label": sample.get("answer_label"),
        "meta": sample.get("meta", {}),
        "model_name": MODEL_NAME,
        "quantization": "4-bit NF4",
        "max_length": MAX_LENGTH,
        "hidden_state_layers": LAYER_IDS,
        "pooling": ["mean", "last_token"],
        "token_ids": token_ids,
        "token_offsets": offsets,
        "step_boundaries": char_boundaries,
        "step_token_indices": step_token_indices,

        # NEW FIELD
        "question_hidden_state": question_states,

        # Existing field
        "step_hidden_states": step_states,
    }

    del result, inputs, encoded
    return artifact


def source_output_dir(source):
    path = OUTPUT_ROOT / source
    path.mkdir(parents=True, exist_ok=True)
    return path

def discover_completed(source):
    folder = source_output_dir(source)
    completed = set()
    for shard_path in sorted(folder.glob("shard_*.pt")):
        index_path = shard_path.with_suffix(".json")
        if index_path.exists():
            index = json.loads(index_path.read_text(encoding="utf-8"))
        else:
            payload = torch.load(shard_path, map_location="cpu", weights_only=False)
            index = {"sample_ids": [a["sample_id"] for a in payload["records"]]}
            index_path.write_text(json.dumps(index), encoding="utf-8")
        completed.update(index["sample_ids"])
    return completed

def save_shard(source, records, shard_number):
    folder = source_output_dir(source)
    shard_path = folder / f"shard_{shard_number:06d}.pt"
    temp_path = folder / f"shard_{shard_number:06d}.tmp"
    torch.save({"source": source, "records": records}, temp_path)
    temp_path.replace(shard_path)
    index_path = shard_path.with_suffix(".json")
    index_path.write_text(
        json.dumps({"sample_ids": [r["sample_id"] for r in records]}),
        encoding="utf-8",
    )
    return shard_path

print("Extraction functions ready.")

Extraction functions ready.


In [41]:
for layer, poolings in test_artifact["question_hidden_state"].items():
    for pooling, tensor in poolings.items():
        print(
            f"Layer {layer} | {pooling}",
            "all_zero =", torch.count_nonzero(tensor).item() == 0,
            "nonzero_values =", torch.count_nonzero(tensor).item(),
            "min =", tensor.min().item(),
            "max =", tensor.max().item(),
            "mean_abs =", tensor.abs().float().mean().item(),
        )

Layer -8 | last_token all_zero = False nonzero_values = 4095 min = -20.296875 max = 69.6875 mean_abs = 3.9419124126434326
Layer -8 | mean all_zero = False nonzero_values = 4096 min = -113.0 max = 726.0 mean_abs = 1.9017024040222168
Layer -4 | last_token all_zero = False nonzero_values = 4094 min = -42.59375 max = 152.875 mean_abs = 6.741841793060303
Layer -4 | mean all_zero = False nonzero_values = 4096 min = -116.0 max = 891.5 mean_abs = 2.8629722595214844
Layer -1 | last_token all_zero = False nonzero_values = 4096 min = -72.9375 max = 41.125 mean_abs = 1.2551285028457642
Layer -1 | mean all_zero = False nonzero_values = 4096 min = -55.0 max = 35.0625 mean_abs = 0.5208194255828857


## 6. Extract all three datasets in resumable batches

In [23]:
# ============================================================
# ReTrace: Resume existing shards + prioritize HaluEval
# Target: 5,000 completed samples per dataset
# ============================================================

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_PER_DATASET = 5000

failures_path = OUTPUT_ROOT / "failed_samples.jsonl"

# Track failed samples separately for each dataset.
failure_ids_by_source = {}

if failures_path.exists():
    with open(failures_path, encoding="utf-8") as f:
        for line in f:
            try:
                record = json.loads(line)
                source_name = record["source"]
                sid = record["sample_id"]

                failure_ids_by_source.setdefault(
                    source_name, set()
                ).add(sid)

            except (json.JSONDecodeError, KeyError):
                pass

# Discover already completed samples from existing shards.
completed_by_source = {
    source: discover_completed(source)
    for source in SOURCE_FILES
}

completed_ids = set().union(
    *completed_by_source.values()
) if completed_by_source else set()

print("Existing completed samples:")
for source, ids in completed_by_source.items():
    print(f"  {source}: {len(ids)} / {TARGET_PER_DATASET}")

print("\nPreviously failed samples:")
for source in SOURCE_FILES:
    print(
        f"  {source}: "
        f"{len(failure_ids_by_source.get(source, set()))}"
    )

# Put HaluEval first without assuming the exact source key spelling.
def is_halueval(source, filename):
    return (
        "halueval" in source.lower().replace("_", "").replace("-", "")
        or "halueval" in filename.lower().replace("_", "").replace("-", "")
    )

source_order = sorted(
    SOURCE_FILES,
    key=lambda source: (
        0 if is_halueval(source, SOURCE_FILES[source]) else 1
    )
)

print("\nProcessing order:")
for source in source_order:
    print(f"  {source}: {SOURCE_FILES[source]}")

print("\nNew-sample budget this run:", MAX_NEW_SAMPLES_PER_RUN)

processed_this_run = 0
failed_this_run = 0
run_started = time.time()

for source in source_order:

    # Preserve the configured total compute budget for this session.
    if processed_this_run >= MAX_NEW_SAMPLES_PER_RUN:
        print("\nPer-run budget reached. Stopping safely.")
        break

    source_done = completed_by_source[source]

    # Do not redo completed work or repeatedly retry logged failures.
    attempted_for_source = (
        set(source_done)
        | failure_ids_by_source.get(source, set())
    )

    if len(source_done) >= TARGET_PER_DATASET:
        print(
            f"\nSkipping {source}: target already reached "
            f"({len(source_done)} completed)."
        )
        continue

    folder = source_output_dir(source)
    folder.mkdir(parents=True, exist_ok=True)

    shard_numbers = [
        int(p.stem.split("_")[-1])
        for p in folder.glob("shard_*.pt")
        if p.stem.split("_")[-1].isdigit()
    ]

    next_shard = max(shard_numbers, default=-1) + 1
    pending = []

    data_path = DATA_DIR / "processed" / SOURCE_FILES[source]

    if not data_path.exists():
        raise FileNotFoundError(
            f"Dataset file missing for {source}: {data_path}"
        )

    print(
        f"\nStarting {source}: "
        f"{len(source_done)} / {TARGET_PER_DATASET} completed"
    )

    with open(data_path, encoding="utf-8") as f:
        for line in f:

            if processed_this_run >= MAX_NEW_SAMPLES_PER_RUN:
                break

            # Count pending samples as well, so we don't overshoot
            # the dataset target while a shard is being assembled.
            if len(source_done) + len(pending) >= TARGET_PER_DATASET:
                break

            sample = json.loads(line)
            sid = sample["sample_id"]

            if sid in attempted_for_source:
                continue

            split_name = split_by_id.get(sid)

            if split_name is None:
                raise KeyError(
                    f"No split assignment for {sid} in {source}"
                )

            try:
                artifact = extract_trajectory(sample, split_name)
                pending.append(artifact)

                attempted_for_source.add(sid)
                processed_this_run += 1

                if len(pending) >= SHARD_SIZE:
                    saved = save_shard(
                        source, pending, next_shard
                    )

                    saved_ids = {
                        a["sample_id"] for a in pending
                    }

                    source_done.update(saved_ids)
                    completed_ids.update(saved_ids)

                    print(
                        f"{source}: saved {len(pending)} samples "
                        f"-> {saved.name}; "
                        f"dataset total={len(source_done)}; "
                        f"this run={processed_this_run}"
                    )

                    pending = []
                    next_shard += 1

            except Exception as exc:
                with open(
                    failures_path, "a", encoding="utf-8"
                ) as log:
                    log.write(json.dumps({
                        "sample_id": sid,
                        "source": source,
                        "error_type": type(exc).__name__,
                        "error": str(exc)[:1000],
                    }) + "\n")

                failure_ids_by_source.setdefault(
                    source, set()
                ).add(sid)

                attempted_for_source.add(sid)
                failed_this_run += 1

                print(
                    f"FAILED {source}/{sid}: "
                    f"{type(exc).__name__}: {str(exc)[:180]}"
                )

                del exc

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                gc.collect()

    # Save the final partial shard, including when the budget
    # or the dataset target is reached.
    if pending:
        saved = save_shard(source, pending, next_shard)

        saved_ids = {a["sample_id"] for a in pending}
        source_done.update(saved_ids)
        completed_ids.update(saved_ids)

        print(
            f"{source}: saved final partial shard "
            f"({len(pending)} samples) -> {saved.name}; "
            f"dataset total={len(source_done)}"
        )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()

    print(
        f"Finished {source}: "
        f"{len(source_done)} / {TARGET_PER_DATASET} completed"
    )

# Save a summary of this session.
summary = {
    "model": MODEL_NAME,
    "layers": LAYER_IDS,
    "pooling": ["mean", "last_token"],
    "max_length": MAX_LENGTH,
    "shard_size": SHARD_SIZE,
    "target_per_dataset": TARGET_PER_DATASET,
    "processing_order": source_order,
    "completed_by_source": {
        source: len(ids)
        for source, ids in completed_by_source.items()
    },
    "remaining_to_target": {
        source: max(
            0, TARGET_PER_DATASET - len(ids)
        )
        for source, ids in completed_by_source.items()
    },
    "processed_this_run": processed_this_run,
    "failed_this_run": failed_this_run,
    "completed_total": len(completed_ids),
    "failed_total": sum(
        len(ids) for ids in failure_ids_by_source.values()
    ),
    "elapsed_minutes": round(
        (time.time() - run_started) / 60, 2
    ),
}

(
    OUTPUT_ROOT / "progress_summary.json"
).write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("\nFINAL SESSION SUMMARY")
print(json.dumps(summary, indent=2))

Existing completed samples:
  processbench: 0 / 5000
  prm800k: 0 / 5000
  halueval: 0 / 5000

Previously failed samples:
  processbench: 0
  prm800k: 0
  halueval: 0

Processing order:
  halueval: halueval.jsonl
  processbench: processbench.jsonl
  prm800k: prm800k.jsonl

New-sample budget this run: 5000

Starting halueval: 0 / 5000 completed
halueval: saved 100 samples -> shard_000000.pt; dataset total=100; this run=100
halueval: saved 100 samples -> shard_000001.pt; dataset total=200; this run=200
halueval: saved 100 samples -> shard_000002.pt; dataset total=300; this run=300
halueval: saved 100 samples -> shard_000003.pt; dataset total=400; this run=400
halueval: saved 100 samples -> shard_000004.pt; dataset total=500; this run=500
halueval: saved 100 samples -> shard_000005.pt; dataset total=600; this run=600
halueval: saved 100 samples -> shard_000006.pt; dataset total=700; this run=700
halueval: saved 100 samples -> shard_000007.pt; dataset total=800; this run=800
halueval: save

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
failures_path = OUTPUT_ROOT / "failed_samples.jsonl"
failure_ids = set()
if failures_path.exists():
    with open(failures_path, encoding="utf-8") as f:
        for line in f:
            try:
                failure_ids.add(json.loads(line)["sample_id"])
            except (json.JSONDecodeError, KeyError):
                pass

completed_by_source = {source: discover_completed(source) for source in SOURCE_FILES}
completed_ids = set().union(*completed_by_source.values()) if completed_by_source else set()
attempted_ids = completed_ids | failure_ids
print("Already completed:", len(completed_ids))
print("Previously failed and logged:", len(failure_ids))
print("New-sample budget this run:", MAX_NEW_SAMPLES_PER_RUN)

processed_this_run = 0
failed_this_run = 0
run_started = time.time()

for source, filename in SOURCE_FILES.items():
    if processed_this_run >= MAX_NEW_SAMPLES_PER_RUN:
        break

    source_done = completed_by_source[source]
    folder = source_output_dir(source)
    shard_numbers = [
        int(p.stem.split("_")[-1])
        for p in folder.glob("shard_*.pt")
        if p.stem.split("_")[-1].isdigit()
    ]
    next_shard = max(shard_numbers, default=-1) + 1
    pending = []

    data_path = DATA_DIR / "processed" / filename
    with open(data_path, encoding="utf-8") as f:
        for line in f:
            if processed_this_run >= MAX_NEW_SAMPLES_PER_RUN:
                break
            sample = json.loads(line)
            sid = sample["sample_id"]
            if sid in attempted_ids:
                continue
            split_name = split_by_id.get(sid)
            if split_name is None:
                raise KeyError(f"No split assignment for {sid}")

            try:
                artifact = extract_trajectory(sample, split_name)
                pending.append(artifact)
                attempted_ids.add(sid)
                processed_this_run += 1
                if len(pending) >= SHARD_SIZE:
                    saved = save_shard(source, pending, next_shard)
                    source_done.update(a["sample_id"] for a in pending)
                    completed_ids.update(a["sample_id"] for a in pending)
                    print(f"{source}: saved {len(pending)} samples -> {saved.name}; total this run={processed_this_run}")
                    pending = []
                    next_shard += 1
            except Exception as exc:
                with open(failures_path, "a", encoding="utf-8") as log:
                    log.write(json.dumps({
                        "sample_id": sid,
                        "source": source,
                        "error_type": type(exc).__name__,
                        "error": str(exc)[:1000],
                    }) + "\n")
                failure_ids.add(sid)
                attempted_ids.add(sid)
                failed_this_run += 1
                print(f"FAILED {sid}: {type(exc).__name__}: {str(exc)[:180]}")
                del exc
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                gc.collect()

    if pending:
        saved = save_shard(source, pending, next_shard)
        source_done.update(a["sample_id"] for a in pending)
        completed_ids.update(a["sample_id"] for a in pending)
        print(f"{source}: saved final partial shard ({len(pending)} samples) -> {saved.name}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

summary = {
    "model": MODEL_NAME,
    "layers": LAYER_IDS,
    "pooling": ["mean", "last_token"],
    "max_length": MAX_LENGTH,
    "shard_size": SHARD_SIZE,
    "processed_this_run": processed_this_run,
    "failed_this_run": failed_this_run,
    "completed_total": len(completed_ids),
    "failed_total": len(failure_ids),
    "expected_total": len(split_by_id),
    "remaining_unattempted_estimate": max(0, len(split_by_id) - len(completed_ids) - len(failure_ids)),
    "elapsed_minutes": round((time.time() - run_started) / 60, 2),
}
(OUTPUT_ROOT / "progress_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

Already completed: 5000
Previously failed and logged: 0
New-sample budget this run: 5000
prm800k: saved 100 samples -> shard_000016.pt; total this run=100
prm800k: saved 100 samples -> shard_000017.pt; total this run=200
prm800k: saved 100 samples -> shard_000018.pt; total this run=300
prm800k: saved 100 samples -> shard_000019.pt; total this run=400
prm800k: saved 100 samples -> shard_000020.pt; total this run=500
prm800k: saved 100 samples -> shard_000021.pt; total this run=600
prm800k: saved 100 samples -> shard_000022.pt; total this run=700
prm800k: saved 100 samples -> shard_000023.pt; total this run=800
prm800k: saved 100 samples -> shard_000024.pt; total this run=900
prm800k: saved 100 samples -> shard_000025.pt; total this run=1000
prm800k: saved 100 samples -> shard_000026.pt; total this run=1100
prm800k: saved 100 samples -> shard_000027.pt; total this run=1200
prm800k: saved 100 samples -> shard_000028.pt; total this run=1300
prm800k: saved 100 samples -> shard_000029.pt; to

## 7. Verify saved shards and report progress

In [24]:
import json
import torch

verification = {}
all_completed = set()

for source in SOURCE_FILES:
    folder = source_output_dir(source)
    source_ids = set()
    shard_count = 0
    for shard_path in sorted(folder.glob("shard_*.pt")):
        payload = torch.load(shard_path, map_location="cpu", weights_only=False)
        records = payload["records"]
        assert records, f"Empty shard: {shard_path}"
        for artifact in records:
            assert artifact["source"] == source
            assert len(artifact["steps"]) == len(artifact["step_boundaries"])
            assert len(artifact["steps"]) == len(artifact["step_token_indices"])
            assert artifact["step_labels"] is None or len(artifact["step_labels"]) == len(artifact["steps"])
            for layer_id in LAYER_IDS:
                states = artifact["step_hidden_states"][layer_id]
                expected_shape = (len(artifact["steps"]), 4096)
                assert tuple(states["mean"].shape) == expected_shape
                assert tuple(states["last_token"].shape) == expected_shape
                assert torch.isfinite(states["mean"]).all().item()
                assert torch.isfinite(states["last_token"]).all().item()
            source_ids.add(artifact["sample_id"])
        shard_count += 1
    verification[source] = {"completed_samples": len(source_ids), "shards": shard_count}
    all_completed.update(source_ids)

failed_count = 0
if failures_path.exists():
    with open(failures_path, encoding="utf-8") as f:
        failed_count = sum(1 for line in f if line.strip())

final_status = {
    "source_counts": verification,
    "completed_unique_samples": len(all_completed),
    "failed_logged_samples": failed_count,
    "expected_samples": len(split_by_id),
    "remaining_not_completed_or_failed": max(0, len(split_by_id) - len(all_completed) - failed_count),
    "output_folder": str(OUTPUT_ROOT),
}
(OUTPUT_ROOT / "verification_summary.json").write_text(json.dumps(final_status, indent=2), encoding="utf-8")
print(json.dumps(final_status, indent=2))
print("Hidden-state artifacts are stored on Google Drive.")

{
  "source_counts": {
    "processbench": {
      "completed_samples": 0,
      "shards": 0
    },
    "prm800k": {
      "completed_samples": 0,
      "shards": 0
    },
    "halueval": {
      "completed_samples": 5000,
      "shards": 50
    }
  },
  "completed_unique_samples": 5000,
  "failed_logged_samples": 0,
  "expected_samples": 139682,
  "remaining_not_completed_or_failed": 134682,
  "output_folder": "/content/drive/MyDrive/ReTrace_GPU_Work/hidden_states_full"
}
Hidden-state artifacts are stored on Google Drive.


In [28]:
from pathlib import Path

root = Path("/content/drive/MyDrive/ReTrace/hidden_states_full")

for source in ["processbench", "prm800k", "halueval"]:
    folder = root / source
    shards = sorted(folder.glob("shard_*.pt")) if folder.exists() else []
    indexes = sorted(folder.glob("shard_*.json")) if folder.exists() else []

    print(f"\n{source}")
    print("  Shards:", len(shards))
    print("  Index files:", len(indexes))
    print("  First shard:", shards[0].name if shards else "None")
    print("  Last shard:", shards[-1].name if shards else "None")


processbench
  Shards: 34
  Index files: 34
  First shard: shard_000000.pt
  Last shard: shard_000033.pt

prm800k
  Shards: 66
  Index files: 66
  First shard: shard_000000.pt
  Last shard: shard_000065.pt

halueval
  Shards: 50
  Index files: 50
  First shard: shard_000000.pt
  Last shard: shard_000049.pt


In [29]:
import torch
from pathlib import Path

root = Path(
    "/content/drive/MyDrive/ReTrace/hidden_states_full"
)

for source in ["processbench", "prm800k", "halueval"]:
    folder = root / source
    shards = sorted(folder.glob("shard_*.pt"))

    print(f"\n{'=' * 50}")
    print(f"Dataset: {source}")
    print("Shard count:", len(shards))

    if not shards:
        print("No shards found")
        continue

    payload = torch.load(
        shards[0],
        map_location="cpu",
        weights_only=False
    )

    print("Payload type:", type(payload))

    if not isinstance(payload, dict):
        print("Unexpected payload format")
        continue

    print("Top-level keys:", list(payload.keys()))

    records = payload.get("records", [])
    print("Records in first shard:", len(records))

    if records:
        record = records[0]
        print("Record keys:", list(record.keys()))

        for key, value in record.items():
            if isinstance(value, torch.Tensor):
                print(
                    f"{key}: shape={tuple(value.shape)}, "
                    f"dtype={value.dtype}"
                )
            elif isinstance(value, (list, tuple)):
                print(f"{key}: sequence length={len(value)}")
            else:
                print(f"{key}: {type(value).__name__}")


Dataset: processbench
Shard count: 34
Payload type: <class 'dict'>
Top-level keys: ['source', 'records']
Records in first shard: 100
Record keys: ['sample_id', 'source', 'problem_id', 'split', 'question', 'steps', 'step_labels', 'first_error', 'answer', 'answer_label', 'meta', 'model_name', 'quantization', 'max_length', 'hidden_state_layers', 'pooling', 'token_ids', 'token_offsets', 'step_boundaries', 'step_token_indices', 'step_hidden_states']
sample_id: str
source: str
problem_id: str
split: str
question: str
steps: sequence length=4
step_labels: sequence length=4
first_error: int
answer: NoneType
answer_label: int
meta: dict
model_name: str
quantization: str
max_length: int
hidden_state_layers: sequence length=3
pooling: sequence length=2
token_ids: shape=(443,), dtype=torch.int64
token_offsets: sequence length=443
step_boundaries: sequence length=4
step_token_indices: sequence length=4
step_hidden_states: dict

Dataset: prm800k
Shard count: 66
Payload type: <class 'dict'>
Top-leve

In [32]:
from pathlib import Path
import torch

ROOT = Path(
    "/content/drive/MyDrive/ReTrace/hidden_states_full"
)

for source in ["processbench", "prm800k", "halueval"]:
    shard = sorted((ROOT / source).glob("shard_*.pt"))[0]
    payload = torch.load(shard, map_location="cpu", weights_only=False)

    exact_matches = 0
    total_steps = 0

    print(f"\n{source.upper()}")

    for record in payload["records"][:100]:
        offsets = record["token_offsets"]
        boundaries = record["step_boundaries"]
        actual_indices = record["step_token_indices"]

        for (start, end), actual in zip(boundaries, actual_indices):
            # Character offsets overlapping the step's character span.
            expected = [
                i for i, (a, b) in enumerate(offsets)
                if a < end and b > start
            ]

            actual = list(actual)
            total_steps += 1

            if actual == expected:
                exact_matches += 1

    print("Steps checked:", total_steps)
    print("Exact boundary/token matches:", exact_matches)
    print(
        "Match rate:",
        round(100 * exact_matches / total_steps, 2)
        if total_steps else 0,
        "%"
    )


PROCESSBENCH
Steps checked: 546
Exact boundary/token matches: 546
Match rate: 100.0 %

PRM800K
Steps checked: 1503
Exact boundary/token matches: 1503
Match rate: 100.0 %

HALUEVAL
Steps checked: 100
Exact boundary/token matches: 100
Match rate: 100.0 %


In [42]:

import json
import torch
from pathlib import Path
from collections import Counter, defaultdict

ROOT = Path(
    "/content/drive/MyDrive/ReTrace/hidden_states_full"
)

EXPECTED_COUNTS = {
    "processbench": 3400,
    "prm800k": 6600,
    "halueval": 5000,
}
EXPECTED_LAYERS = {-8, -4, -1}
EXPECTED_POOLING = {"mean", "last_token"}
EXPECTED_DIM = 4096

errors = []
warnings = []
all_ids = set()
duplicate_ids = []
split_by_problem = defaultdict(set)
dataset_counts = Counter()
total_shards = 0
total_indexes = 0
total_steps = 0
total_tensors_checked = 0
missing_question_state = 0


def report_error(message):
    errors.append(message)


for source, expected_count in EXPECTED_COUNTS.items():
    folder = ROOT / source
    shards = sorted(folder.glob("shard_*.pt"))
    indexes = sorted(folder.glob("shard_*.json"))

    print(f"\n{'=' * 55}")
    print(source.upper())
    print("Shards:", len(shards))
    print("Indexes:", len(indexes))

    if not folder.exists():
        report_error(f"{source}: directory missing")
        continue

    if len(shards) != len(indexes):
        report_error(f"{source}: shard/index count mismatch")

    source_ids = set()

    for shard_path in shards:
        total_shards += 1
        index_path = shard_path.with_suffix(".json")

        try:
            payload = torch.load(
                shard_path,
                map_location="cpu",
                weights_only=False,
            )
        except Exception as exc:
            report_error(
                f"Cannot load {shard_path.name}: {exc}"
            )
            continue

        if not isinstance(payload, dict):
            report_error(f"{shard_path.name}: invalid payload")
            continue

        if payload.get("source") != source:
            report_error(
                f"{shard_path.name}: incorrect source metadata"
            )

        records = payload.get("records")
        if not isinstance(records, list):
            report_error(f"{shard_path.name}: records missing")
            continue

        # Verify the corresponding index.
        if not index_path.exists():
            report_error(f"Missing index: {index_path.name}")
            index_ids = None
        else:
            total_indexes += 1
            try:
                index = json.loads(
                    index_path.read_text(encoding="utf-8")
                )
                index_ids = index["sample_ids"]

                if len(index_ids) != len(set(index_ids)):
                    report_error(
                        f"{index_path.name}: duplicate IDs in index"
                    )

                record_ids = [
                    r.get("sample_id") for r in records
                ]

                if index_ids != record_ids:
                    report_error(
                        f"{index_path.name}: index IDs do not "
                        "match shard record IDs and order"
                    )
            except Exception as exc:
                report_error(
                    f"Invalid index {index_path.name}: {exc}"
                )
                index_ids = None

        for record in records:
            sid = record.get("sample_id")

            if not sid:
                report_error(
                    f"{shard_path.name}: missing sample_id"
                )
                continue

            dataset_counts[source] += 1

            if sid in source_ids:
                report_error(
                    f"{source}: duplicate sample ID {sid}"
                )
            source_ids.add(sid)

            global_key = (source, sid)
            if global_key in all_ids:
                duplicate_ids.append(global_key)
            all_ids.add(global_key)

            # Check essential fields.
            required = {
                "question", "steps", "token_ids", "token_offsets",
                "step_boundaries", "step_token_indices",
                "step_hidden_states", "hidden_state_layers",
                "pooling", "split",
            }
            missing = required - record.keys()

            if missing:
                report_error(
                    f"{sid}: missing fields {sorted(missing)}"
                )
                continue

            if "question_hidden_state" not in record:
                missing_question_state += 1

            steps = record["steps"]
            offsets = record["token_offsets"]
            boundaries = record["step_boundaries"]
            step_indices = record["step_token_indices"]
            token_ids = record["token_ids"]
            n_tokens = len(token_ids)
            n_steps = len(steps)

            if not (
                len(boundaries)
                == len(step_indices)
                == n_steps
            ):
                report_error(
                    f"{sid}: steps/boundaries/index lengths differ"
                )
                continue

            labels = record.get("step_labels")
            if labels is not None and len(labels) != n_steps:
                report_error(
                    f"{sid}: step_labels length differs from steps"
                )

            # Check character boundaries against token offsets.
            for step_i, ((start, end), indices) in enumerate(
                zip(boundaries, step_indices)
            ):
                if start < 0 or end <= start:
                    report_error(
                        f"{sid}: malformed boundary for step {step_i}"
                    )
                    continue

                expected_indices = [
                    i for i, (a, b) in enumerate(offsets)
                    if b > a and a < end and b > start
                ]

                actual_indices = list(indices)

                if actual_indices != expected_indices:
                    report_error(
                        f"{sid}: boundary/index mismatch at step "
                        f"{step_i}"
                    )

                if any(i < 0 or i >= n_tokens for i in actual_indices):
                    report_error(
                        f"{sid}: out-of-range token index"
                    )

            # Check hidden-state layers, pooling and tensor contents.
            states = record["step_hidden_states"]

            if set(states.keys()) != EXPECTED_LAYERS:
                report_error(f"{sid}: unexpected hidden-state layers")

            if set(record["pooling"]) != EXPECTED_POOLING:
                report_error(f"{sid}: unexpected pooling metadata")

            for layer in EXPECTED_LAYERS:
                if layer not in states:
                    continue

                if set(states[layer].keys()) != EXPECTED_POOLING:
                    report_error(
                        f"{sid}: incorrect pooling keys at layer {layer}"
                    )
                    continue

                for pooling in EXPECTED_POOLING:
                    tensor = states[layer][pooling]
                    total_tensors_checked += 1

                    if not isinstance(tensor, torch.Tensor):
                        report_error(f"{sid}: non-tensor state")
                        continue

                    if tuple(tensor.shape) != (n_steps, EXPECTED_DIM):
                        report_error(
                            f"{sid}: incorrect shape for layer "
                            f"{layer}/{pooling}"
                        )

                    if not torch.isfinite(tensor).all().item():
                        report_error(
                            f"{sid}: non-finite tensor at "
                            f"{layer}/{pooling}"
                        )

            total_steps += n_steps

            problem_id = record.get("problem_id")
            split = record.get("split")
            if problem_id is not None and split is not None:
                split_by_problem[(source, problem_id)].add(split)

    if len(source_ids) != expected_count:
        report_error(
            f"{source}: expected {expected_count} unique samples, "
            f"found {len(source_ids)}"
        )

# Detect the same problem assigned to multiple splits.
split_leaks = [
    key for key, splits in split_by_problem.items()
    if len(splits) > 1
]
if split_leaks:
    report_error(
        f"{len(split_leaks)} source/problem IDs appear in multiple splits"
    )

print("\n" + "=" * 55)
print("FINAL AUDIT")
print("Samples:", dict(dataset_counts))
print("Total samples:", sum(dataset_counts.values()))
print("Unique source/sample IDs:", len(all_ids))
print("Shards:", total_shards)
print("Indexes:", total_indexes)
print("Steps checked:", total_steps)
print("Tensors checked:", total_tensors_checked)
print("Records missing question_hidden_state:", missing_question_state)
print("Cross-split problem IDs:", len(split_leaks))
print("Errors:", len(errors))

for error in errors[:30]:
    print("ERROR:", error)

if len(errors) > 30:
    print(f"... and {len(errors) - 30} additional errors")

print(
    "\nRESULT:",
    "PASS — structural audit clean"
    if not errors
    else "NOT READY — investigate errors above"
)



PROCESSBENCH
Shards: 34
Indexes: 34

PRM800K
Shards: 66
Indexes: 66

HALUEVAL
Shards: 50
Indexes: 50

FINAL AUDIT
Samples: {'processbench': 3400, 'prm800k': 6600, 'halueval': 5000}
Total samples: 15000
Unique source/sample IDs: 15000
Shards: 150
Indexes: 150
Steps checked: 109741
Tensors checked: 90000
Records missing question_hidden_state: 15000
Cross-split problem IDs: 0
Errors: 0

RESULT: PASS — structural audit clean
